# Task 1 – NAV History Cleaning

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")
print(nav.shape)
print(nav.info())
print(nav.head())

In [ ]:
nav["date"] = pd.to_datetime(nav["date"])

nav = (nav.sort_values(["amfi_code", "date"]).drop_duplicates())
invalid_nav = nav[nav["nav"] <= 0]
print("Duplicate Rows:", nav.duplicated().sum())
print("Invalid NAV Rows:", len(invalid_nav))

In [ ]:
cleaned_list = []

for amfi_code, group in nav.groupby("amfi_code"):
    full_dates = pd.date_range(start=group["date"].min(),end=group["date"].max(),freq="D")
    group = (group.set_index("date").reindex(full_dates))
    group["amfi_code"] = amfi_code
    group["nav"] = group["nav"].ffill()
    group = (group.reset_index().rename(columns={"index": "date"}))
    cleaned_list.append(group)
clean_nav = pd.concat(cleaned_list, ignore_index=True)

In [ ]:
print("Final Shape:", clean_nav.shape)
print(clean_nav.isnull().sum())
clean_nav.to_csv("../data/processed/clean_nav_history.csv",index=False)
print("Saved")

## Summary

The NAV History dataset was cleaned and validated. The date column was converted to datetime format and datas were sorted by AMFI code and date. Nav values are verified to be greater then zero.

Missing dates, including weekends and holidays, were added for each fund, and NAV values were forward-filled using the latest available data. This increased the dataset size from 46,000 to 64,320 records.

The cleaned dataset was saved as data/processed/clean_nav_history.csv.

# Task 2 – Investor Transactions Cleaning

In [ ]:
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
print("Shape:", transactions.shape)

In [ ]:
transactions.info()
print("\nMissing Values:")
print(transactions.isnull().sum())

In [ ]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.title()
    .replace({"Sip": "SIP"})
)
transactions = transactions.drop_duplicates()

In [ ]:
invalid_amount = transactions[transactions["amount_inr"] <= 0]
print("Invalid Amount Rows:", len(invalid_amount))
print("\nTransaction Types:")
print(transactions["transaction_type"].unique())
print("\nKYC Status:")
print(transactions["kyc_status"].unique())
print("\nKYC Counts:")
print(transactions["kyc_status"].value_counts())
print("\nDuplicate Rows:",transactions.duplicated().sum())

In [ ]:
transactions.to_csv("../data/processed/clean_investor_transactions.csv",index=False)
print("Saved")

## Summary
The investor transactions dataset was cleaned by converting transaction dates to datetime format, standardizing transaction type values, and removing duplicate records. Data validation ensured that all transaction amounts were positive and that KYC status values were limited to the expected categories: Verified and Pending.

The cleaned dataset was then saved

# Task 3 – Scheme Performance Cleaning

In [ ]:
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
print("Shape:", performance.shape)
performance.head()

In [ ]:
performance.info()
print("\nMissing Values:")
print(performance.isnull().sum())

In [ ]:
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct"
]

for col in return_cols:
    performance[col] = pd.to_numeric(
        performance[col],
        errors="coerce"
    )

performance = performance.drop_duplicates()

In [ ]:
invalid_expense = performance[(performance["expense_ratio_pct"] < 0.1) |
    (performance["expense_ratio_pct"] > 2.5)]
return_anomalies = performance[(performance["return_1yr_pct"] > 100) |
    (performance["return_1yr_pct"] < -100)]

negative_beta = performance[performance["beta"] < 0]

print("Expense Ratio Anomalies:", len(invalid_expense))
print("Return Anomalies:", len(return_anomalies))
print("Negative Beta Records:", len(negative_beta))
print("Duplicate Rows:", performance.duplicated().sum())

print("\nExpense Ratio Range:")
print(performance["expense_ratio_pct"].min(),"to",performance["expense_ratio_pct"].max())

In [ ]:
print("Final Shape:")
print(performance.shape)
print("\nMissing Values:")
print(performance.isnull().sum())

In [ ]:
performance.to_csv("../data/processed/clean_scheme_performance.csv",index=False)

print("Saved")

## Summary
The scheme performance dataset was validated to ensure data accuracy and consistency. Return-related fields were confirmed as numeric, duplicate records were removed, and key performance indicators were checked for unusual values.

Expense ratios were verified to be within the expected industry range, while return metrics and beta values were reviewed for anomalies. The dataset successfully passed all validation checks and was saved